In [0]:
# 07_mlflow_intro.py - MLflow Experiment Tracking dans Databricks Free Edition

import mlflow
import json, tempfile, os
from pyspark.sql import functions as F

In [0]:
# 1. Charger les données depuis Delta Lake
df = spark.read.format("delta").load(
    "/Volumes/workspace/default/raw_data/sirene_clean_delta"
)
print(f"Lignes chargées : {df.count():,}")
df.printSchema()

Lignes chargées : 134,666
root
 |-- siren: integer (nullable = true)
 |-- nic: integer (nullable = true)
 |-- siret: long (nullable = true)
 |-- statut_diffusion: string (nullable = true)
 |-- date_creation_etab: string (nullable = true)
 |-- tranche_effectif: string (nullable = true)
 |-- activite_principale_etab: string (nullable = true)
 |-- etablissement_siege: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- commune: string (nullable = true)
 |-- code_commune: integer (nullable = true)
 |-- code_departement: integer (nullable = true)
 |-- departement: string (nullable = true)
 |-- code_region: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- etat_admin_etab: string (nullable = true)
 |-- date_fermeture_etab: date (nullable = true)
 |-- denomination_unite_legale: string (nullable = true)
 |-- categorie_entreprise: string (nullable = true)
 |-- etat_admin_ul: string (nullable = true)
 |-- caractere_employeur: string (nullable = true)
 |-- 

In [0]:
# 2. Créer / sélectionner l'experiment
mlflow.set_experiment("/Users/brucher.alan@gmail.com/M3_SIRENE/sirene_analyse_effectif")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/4027276423216532', creation_time=1785274788787, experiment_id='4027276423216532', last_update_time=1785274808379, lifecycle_stage='active', name='/Users/brucher.alan@gmail.com/M3_SIRENE/sirene_analyse_effectif', tags={'mlflow.experiment.sourceName': '/Users/brucher.alan@gmail.com/M3_SIRENE/sirene_analyse_effectif',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'brucher.alan@gmail.com',
 'mlflow.ownerId': '74238550927607'}>

In [0]:
# 3. Lancer un run de tracking
with mlflow.start_run(run_name="analyse_baseline_m3"):

    # Log des paramètres (strings, config — non chartables)
    mlflow.log_param("source_table", "sirene_clean_delta")
    mlflow.log_param("filtre_etat", "Actif")
    mlflow.log_param("filtre_rgpd", "statut_diffusion != P")
    mlflow.log_param("partitionnement", "categorie_entreprise")

    # Calculer les métriques
    total = df.count()
    sieges = df.filter(F.col("est_siege") == True).count()
    taux_siege = round(sieges / total * 100, 2)
    nb_communes = df.select("commune").distinct().count()

    # Log des métriques (numériques, graphables, comparables entre runs)
    mlflow.log_metric("total_etablissements", total)
    mlflow.log_metric("total_sieges", sieges)
    mlflow.log_metric("taux_siege_pct", taux_siege)
    mlflow.log_metric("nb_communes_distinctes", nb_communes)

    # Log d'un artifact (fichier JSON de résumé)
    summary = {
        "source": "sirene_clean_delta",
        "total_etablissements": total,
        "total_sieges": sieges,
        "taux_siege_pct": taux_siege,
        "nb_communes_distinctes": nb_communes,
    }
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".json", delete=False, encoding="utf-8"
    ) as f:
        json.dump(summary, f, indent=2)
        tmp_path = f.name
    mlflow.log_artifact(tmp_path, "metadata")
    os.unlink(tmp_path)

    print(f"Run loggé : {total:,} établissements · {taux_siege}% de sièges")
    print(f"Communes distinctes : {nb_communes}")

Run loggé : 134,666 établissements · 87.33% de sièges
Communes distinctes : 34


In [0]:
# 4. Consulter les résultats programmatiquement
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("/Users/brucher.alan@gmail.com/M3_SIRENE/sirene_analyse_effectif")
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["start_time DESC"],
    max_results=3,
)
print("\n--- Derniers runs ---")
for r in runs:
    t = r.data.metrics.get("total_etablissements", "—")
    print(f"  {r.info.run_name} · total={t:,}" if isinstance(t, (int, float)) else f"  {r.info.run_name}")


--- Derniers runs ---
  analyse_baseline_m3 · total=134,666.0
  analyse_baseline_m3 · total=134,666.0
